# TNBC Regulatory Pipeline Appyter
**ChEA3 → KEA3 → Enrichr**

This appyter takes a gene set — for example, genes up-regulated in triple-negative breast cancer (TNBC) — and chains together three enrichment tools to build a regulatory hypothesis:

1. **ChEA3** infers which transcription factors are most likely driving the gene set.
2. **KEA3** takes those transcription factors and infers the upstream kinases regulating them.
3. **Enrichr** (LINCS L1000) proposes small-molecule drugs predicted to reverse the original gene signature.

For simplicity, the primary inputs are a gene list and a handful of parameters controlling how many top hits flow through each step. Other parameters are set to reasonable defaults in the cells below — you can download the notebook, change them, and rerun it if you wish.

Links to the full, interactive results on the ChEA3, KEA3, and Enrichr websites are provided at the bottom of each section.

In [2]:
#%%appyter init
from appyter import magic
magic.init(lambda _=globals: _())

In [1]:
import requests
import json
import pandas as pd
from IPython.display import display, HTML, Markdown

In [3]:
%%appyter hide_code

{% do SectionField(
    name='section1',
    title='1. Submit Your Gene List',
    subtitle='Upload a text file containing your gene list or copy and paste your gene list into the text box below (one gene per row). You can also try the default TNBC gene set provided.',
) %}
{% do SectionField(
    name='section2',
    title='2. Pipeline Parameters',
    subtitle='Set a query name and choose how many top hits to carry forward at each step of the pipeline.',
) %}

In [4]:
%%appyter hide_code

{% set gene_list_kind = TabField(
    name='gene_list_kind',
    label='Gene List',
    default='Paste',
    description='Paste or upload your gene list',
    required=True,
    choices={
        'Paste': [
            TextListField(
                name='gene_list_input',
                label='Gene List',
                description='Paste your gene list (one gene per row).',
                default=[
                    'MKI67', 'TOP2A', 'MELK', 'FOXM1', 'CCNB1', 'CDK1', 'BIRC5', 'PLK1',
                    'AURKB', 'BUB1', 'CCNA2', 'CDC20', 'CENPF', 'KIF2C', 'NDC80', 'RRM2',
                    'TYMS', 'PTTG1', 'KIF20A', 'ASPM', 'TPX2', 'PRC1', 'NUSAP1', 'KIF11',
                    'DLGAP5', 'HJURP', 'CDCA8', 'TTK', 'HMMR', 'EXO1', 'RAD51', 'BRCA1',
                    'EZH2', 'MYC', 'E2F1', 'E2F2', 'STAT3', 'EGFR', 'VIM', 'FN1',
                    'SNAI1', 'TWIST1', 'ZEB1', 'CDH2', 'MMP9', 'MMP2', 'VEGFA'
                ],
                section='section1'
            ),
        ],
        'Upload': [
            FileField(
                name='gene_list_filename',
                label='Gene List File',
                default='',
                description='Upload your gene list as a text file (one gene per row).',
                section='section1'
            ),
        ],
    },
    section='section1',
) %}

{% set query_name = StringField(
    name='query_name',
    label='Query Name',
    description='A label for this analysis, sent to each API and used in output titles.',
    default='TNBC_up',
    section='section2'
) %}

{% set top_n_tfs = IntField(
    name='top_n_tfs',
    label='Top Transcription Factors (ChEA3 \u2192 KEA3)',
    description='Number of top-ranked ChEA3 transcription factors to carry forward into the KEA3 kinase analysis.',
    default=10,
    min=1,
    max=25,
    section='section2'
) %}

{% set top_n_kinases = IntField(
    name='top_n_kinases',
    label='Top Kinases to Display',
    default=10,
    min=1,
    max=25,
    section='section2'
) %}

{% set top_n_drugs = IntField(
    name='top_n_drugs',
    label='Top Drug Reversal Candidates to Display',
    default=12,
    min=1,
    max=50,
    section='section2'
) %}

In [ ]:
%%appyter code_exec

{%- if gene_list_kind.raw_value == 'Paste' %}
gene_list_input = {{ gene_list_kind.value[0] }}
{%- else %}
gene_list_filename = {{ gene_list_kind.value[0] }}
{%- endif %}
query_name = {{ query_name }}
top_n_tfs = {{ top_n_tfs }}
top_n_kinases = {{ top_n_kinases }}
top_n_drugs = {{ top_n_drugs }}

In [ ]:
%%appyter code_exec

{%- if gene_list_kind.raw_value == 'Paste' %}
genes = [x.strip() for x in gene_list_input]
{%- else %}
open_gene_list_file = open(gene_list_filename, 'r')
lines = open_gene_list_file.readlines()
genes = [x.strip() for x in lines]
open_gene_list_file.close()
{%- endif %}
genes = [g for g in genes if g]

# Error handling
class NoResults(Exception):
    pass

class APIFailure(Exception):
    pass

print(f'Gene set loaded: {len(genes)} genes')

## Step 1: ChEA3 — Transcription Factor Enrichment
ChEA3 ranks transcription factors by integrating six reference libraries (ChIP-seq, co-expression, and co-occurrence data) and combining them with a mean-rank method. The table below shows the top-ranked transcription factors most likely responsible for regulating the input gene set, along with the genes from your list that overlap each transcription factor's known targets.

In [ ]:
def query_chea3(gene_set, name):
    r = requests.post(
        'https://maayanlab.cloud/chea3/api/enrich/',
        json={'query_name': name, 'gene_set': gene_set},
        headers={'Content-Type': 'application/json'}
    )
    if not r.ok:
        raise APIFailure
    data = r.json()
    if 'Integrated--meanRank' not in data or len(data['Integrated--meanRank']) == 0:
        raise NoResults
    return data

tf_names = []
caption1 = f"**Table 1. Top {top_n_tfs} transcription factors inferred by ChEA3 for `{query_name}`.** Transcription factors are ranked by the Integrated MeanRank method across ChEA3's six reference libraries. Overlapping genes are the members of the input list found among each factor's known target genes."

try:
    chea3 = query_chea3(genes, query_name)
    top_tfs = chea3['Integrated--meanRank'][:top_n_tfs]
    tf_names = [t['TF'] for t in top_tfs]

    chea3_df = pd.DataFrame([
        {
            'Rank': i + 1,
            'Transcription Factor': t['TF'],
            'Score': round(float(t['Score']), 2),
            'Overlapping Genes': ', '.join((t.get('Overlapping_Genes') or [])[:6])
        }
        for i, t in enumerate(top_tfs)
    ])
    display(HTML(f'<strong>Top {top_n_tfs} Transcription Factors (ChEA3)</strong>'))
    display(HTML(chea3_df.to_html(index=False)))
    display(Markdown(caption1))
except APIFailure:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>Unable to retrieve results because of a bad response from the ChEA3 API</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again later.</div>"))
except NoResults:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>No transcription factors were returned for this gene set</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again with a different gene list.</div>"))

## Step 2: KEA3 — Upstream Kinase Enrichment
The transcription factors identified in Step 1 become the input gene set for this step: KEA3 infers which kinases are most likely upstream of them, based on kinase-substrate and protein-protein interaction data integrated across multiple libraries via the Mean Rank method.

In [ ]:
def query_kea3(gene_set, name):
    r = requests.post(
        'https://maayanlab.cloud/kea3/api/enrich/',
        json={'query_name': name, 'gene_set': gene_set}
    )
    if not r.ok:
        raise APIFailure
    data = r.json()
    if not data:
        raise NoResults
    return data

caption2 = f"**Table 2. Top {top_n_kinases} upstream kinases inferred by KEA3 for the transcription factors in Table 1.** Kinases are ranked by KEA3's Integrated MeanRank method. Overlapping genes/proteins are members of the query list found among each kinase's putative substrates."

if not tf_names:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>Skipping KEA3 — no transcription factors available from Step 1</b></div>"))
else:
    try:
        kea3 = query_kea3(tf_names, f'{query_name}_TFs')

        # KEA3 shares its API framework with ChEA3: the response is a DICT keyed
        # by library name (e.g. 'Integrated--meanRank', 'Integrated--topRank',
        # plus one key per individual library), not a list of
        # {'library', 'results'} objects. Each value is a list of per-kinase
        # record dicts.
        mean_rank_key = next(
            (k for k in kea3 if 'meanrank' in k.lower()),
            next(iter(kea3), None)
        )
        top_kinases = kea3.get(mean_rank_key, [])[:top_n_kinases] if mean_rank_key else []
        if not top_kinases:
            raise NoResults

        rows = []
        for i, k in enumerate(top_kinases):
            # Field names vary slightly by library/version, so fall back gracefully.
            name = k.get('TF') or k.get('Kinase') or k.get('kinase') or '?'
            raw_score = k.get('Score', k.get('FET p-value', k.get('FDR', 0)))
            try:
                score = round(float(raw_score), 3)
            except (TypeError, ValueError):
                score = 0.0
            og = k.get('Overlapping_Genes', '')
            if isinstance(og, str):
                genes_overlap = ', '.join(og.split(',')[:6])
            elif isinstance(og, list):
                genes_overlap = ', '.join(og[:6])
            else:
                genes_overlap = ''
            rows.append({'Rank': i + 1, 'Kinase': name, 'Score': score, 'Overlapping Genes': genes_overlap})

        kea3_df = pd.DataFrame(rows)
        display(HTML(f'<strong>Top {top_n_kinases} Upstream Kinases (KEA3 \u00b7 {mean_rank_key})</strong>'))
        display(HTML(kea3_df.to_html(index=False)))
        display(Markdown(caption2))
    except APIFailure:
        display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>Unable to retrieve results because of a bad response from the KEA3 API</b></div>"))
        display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again later.</div>"))
    except NoResults:
        display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>No kinases were returned for this transcription factor set</b></div>"))
        display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again with a different gene list.</div>"))

## Step 3: Enrichr — Drug Reversal Candidates (LINCS L1000)
The original input gene list is uploaded to Enrichr and compared against the LINCS L1000 Chemical Perturbation Consensus Signatures library. Drugs whose signature moves the gene set in the *opposite* direction (marked ↓ reversal) are candidates for reversing the TNBC-associated expression pattern; drugs marked ↑ same push the signature further in the same direction.

In [ ]:
def query_enrichr(gene_set, name):
    r = requests.post(
        'https://maayanlab.cloud/Enrichr/addList',
        files={'list': (None, '\n'.join(gene_set)), 'description': (None, name)}
    )
    if not r.ok:
        raise APIFailure
    data = r.json()
    list_id = data['userListId']
    short_id = data.get('shortId')

    r = requests.get(
        'https://maayanlab.cloud/Enrichr/enrich',
        params={'userListId': list_id, 'backgroundType': 'LINCS_L1000_Chem_Pert_Consensus_Sigs'}
    )
    if not r.ok:
        raise APIFailure
    result = r.json().get('LINCS_L1000_Chem_Pert_Consensus_Sigs', [])
    if len(result) == 0:
        raise NoResults
    return result, short_id

caption3 = f"**Table 3. Top {top_n_drugs} drug reversal candidates from Enrichr (LINCS L1000 Chemical Perturbation Consensus Signatures) for `{query_name}`.** Score is the absolute combined score from the Enrichr enrichment calculation. Direction indicates whether the drug's signature moves the query genes opposite to (reversal) or the same as (same) their behavior in the input list."

enrichr_short_id = None
try:
    drugs_raw, enrichr_short_id = query_enrichr(genes, query_name)
    drugs_raw = drugs_raw[:top_n_drugs]

    rows = []
    for i, d in enumerate(drugs_raw):
        term = d[1] or ''
        drug_name = term.split()[0] if term else '?'
        score = abs(d[4] or 0)
        padj = d[6]
        direction = 'reversal \u2193' if '-dn' in term.lower() else ('same \u2191' if '-up' in term.lower() else '\u2014')
        rows.append({
            'Rank': i + 1,
            'Drug': drug_name,
            'Score': round(score, 1),
            'Direction': direction,
            'Adj. p-value': f'{padj:.2e}' if padj is not None else 'n/a'
        })

    drugs_df = pd.DataFrame(rows)
    display(HTML(f'<strong>Top {top_n_drugs} Drug Reversal Candidates (Enrichr \u00b7 LINCS L1000)</strong>'))
    display(HTML(drugs_df.to_html(index=False)))
    display(Markdown(caption3))
except APIFailure:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>Unable to retrieve results because of a bad response from the Enrichr API</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again later.</div>"))
except NoResults:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>No drug signatures were returned for this gene set</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again with a different gene list.</div>"))

## Link to Enrichr

In [ ]:
# Get a link to the complete enrichment analysis results on the Enrichr website
if enrichr_short_id:
    url = f'https://maayanlab.cloud/Enrichr/enrich?dataset={enrichr_short_id}'
    display(HTML(f"<div style='font-size:1.25rem; padding:1rem 0;'><a href='{url}' target='_blank'>Access the complete enrichment analysis results on the Enrichr website.</a></div>"))
else:
    display(HTML("<div style='font-size:1.5rem; padding:1rem 0;'><b>No Enrichr results available for the current query</b></div>"))
    display(HTML("<div style='font-size:1rem; padding:1rem 0;'>Please try again with a different input list.</div>"))

## Save Full Pipeline Report
Combine the results from all three steps (ChEA3, KEA3, Enrichr) into a single standalone HTML report file that can be downloaded and shared.

In [ ]:
# Save combined results from all three steps into a single HTML report
from datetime import datetime

def _safe_table_html(df_name, caption_name):
    df = globals().get(df_name)
    caption = globals().get(caption_name, '')
    if df is not None:
        return f"<div>{df.to_html(index=False)}</div><p>{caption}</p>"
    else:
        return "<p><i>No results were available for this step.</i></p>"

report_sections = []
report_sections.append("<h2>Step 1: ChEA3 &mdash; Transcription Factor Enrichment</h2>")
report_sections.append(_safe_table_html('chea3_df', 'caption1'))
report_sections.append("<h2>Step 2: KEA3 &mdash; Upstream Kinase Enrichment</h2>")
report_sections.append(_safe_table_html('kea3_df', 'caption2'))
report_sections.append("<h2>Step 3: Enrichr &mdash; Drug Reversal Candidates (LINCS L1000)</h2>")
report_sections.append(_safe_table_html('drugs_df', 'caption3'))

if 'enrichr_short_id' in globals() and enrichr_short_id:
    enrichr_url = f'https://maayanlab.cloud/Enrichr/enrich?dataset={enrichr_short_id}'
    report_sections.append(f"<p><a href='{enrichr_url}' target='_blank'>Access the complete Enrichr results online.</a></p>")

html_report = f"""<html>
<head>
<meta charset="utf-8">
<title>TNBC Regulatory Pipeline Report - {query_name}</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 2rem; color: #222; }}
  table {{ border-collapse: collapse; margin-bottom: 1rem; }}
  th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
  th {{ background-color: #f2f2f2; }}
  h1 {{ border-bottom: 2px solid #333; padding-bottom: 0.5rem; }}
  h2 {{ margin-top: 2rem; color: #333; }}
</style>
</head>
<body>
<h1>TNBC Regulatory Pipeline Report: {query_name}</h1>
<p>Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
{''.join(report_sections)}
</body>
</html>
"""

report_filename = f"{query_name}_pipeline_report.html".replace(' ', '_')
with open(report_filename, 'w') as f:
    f.write(html_report)

display(HTML(f"<div style='font-size:1.25rem; padding:1rem 0;'>Full pipeline report saved to <code>{report_filename}</code></div>"))
display(HTML(f'<div>Download full report: <a href="{report_filename}" target=_blank>{report_filename}</a></div>'))